# 07 MERGE and Upserts (SCD Type 1 and Type 2)

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Use <code>MERGE INTO</code> to insert, update and delete in one atomic operation: basic upserts, conditional clauses, deletes from the source, <code>WHEN NOT MATCHED BY SOURCE</code>, deduplicating the source first, and the two classic dimension patterns, <b>SCD Type 1</b> (overwrite) and <b>SCD Type 2</b> (keep history).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
<code>MERGE</code> is how Silver and Gold tables are kept up to date from changing sources: CDC feeds, daily extracts, late corrections. It makes incremental loads <b>idempotent</b> (re-running doesn't duplicate data). SCD Type 2 is a favourite interview question, and <code>MERGE</code> syntax is heavily tested on the exam. This lesson is <b>interview-critical</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Anatomy of MERGE</b><br>
```sql<br>MERGE INTO target t<br>USING source s<br>ON t.key = s.key                                   -- how rows are matched<br>WHEN MATCHED AND s.op = 'D' THEN DELETE            -- clauses are checked in order<br>WHEN MATCHED THEN UPDATE SET *                     -- or SET t.col = s.col, ...<br>WHEN NOT MATCHED THEN INSERT *                     -- source rows with no target match<br>WHEN NOT MATCHED BY SOURCE THEN DELETE             -- target rows with no source match (optional)<br><code>`</code>
<ul><li>Each target row may match <b>at most one</b> source row, otherwise the <code>MERGE</code> fails. Deduplicate the source first</li><li>Several <code>WHEN MATCHED</code> / <code>WHEN NOT MATCHED</code> clauses are allowed, each with an optional extra condition. The first one that applies wins</li><li><code>UPDATE SET *</code> and <code>INSERT *</code> use columns with the same names</li><li>The whole <code>MERGE</code> is one commit</li></ul>
</div>

| Pattern | What it keeps | MERGE shape |
|---|---|---|
| **Upsert / SCD Type 1** | Only the latest values | `WHEN MATCHED THEN UPDATE`, `WHEN NOT MATCHED THEN INSERT` |
| **SCD Type 2** | Full history: one row per version, with validity dates and a current flag | Close the current row, insert a new current row |
| **Insert-only dedupe** | First occurrence only | `WHEN NOT MATCHED THEN INSERT` only |
| **Full sync** | Exact mirror of the source | Upsert + `WHEN NOT MATCHED BY SOURCE THEN DELETE` |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retailer's customer dimension receives a daily CDC extract: new customers, changed addresses and closed accounts. Marketing wants the <b>current</b> address (Type 1), but finance must report sales by the address valid <b>at the time of the sale</b> (Type 2). One Type 2 <code>MERGE</code> per day keeps both answerable: <code>WHERE is_current</code> gives the latest address, and a join on the validity dates gives the historical one.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a <code>customers</code> target table and a daily changes source with an insert, an update, a delete flag and a no-op.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 target customers, then 4 changes: C2 moves city, C4 is marked for deletion, C5 is new, and C1 is sent unchanged.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

target = f"{DB}.customers_merge"
spark.sql(f"CREATE OR REPLACE TABLE {target} (customer_id STRING, name STRING, city STRING, updated_at DATE)")
spark.sql(f"""
    INSERT INTO {target} VALUES
      ('C1', 'Amal', 'Chennai', DATE'2024-01-01'),
      ('C2', 'John', 'London', DATE'2024-01-01'),
      ('C3', 'Vivek', 'Dubai', DATE'2024-01-01'),
      ('C4', 'Sara', 'Abu Dhabi', DATE'2024-01-01')
""")

changes = spark.createDataFrame(
    [("C1", "Amal", "Chennai", "2024-02-01", "U"),
     ("C2", "John", "Manchester", "2024-02-01", "U"),
     ("C4", "Sara", "Abu Dhabi", "2024-02-01", "D"),
     ("C5", "Priya", "Mumbai", "2024-02-01", "I")],
    "customer_id STRING, name STRING, city STRING, updated_at STRING, op STRING",
).withColumn("updated_at", F.to_date("updated_at"))
changes.createOrReplaceTempView("changes")
spark.table(target).orderBy("customer_id").show()
changes.show()

## 1. A basic upsert (SCD Type 1)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Updates existing customers and inserts new ones, ignoring the <code>op</code> column for now.<br><br>
<b>Why it matters</b><br>This is the most common <code>MERGE</code>: make the target reflect the latest source values, with no duplicates even if the job runs twice. Old values are overwritten, which is SCD Type 1.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
5 customers: C2 in Manchester, C5 added, C4 still present (deletes come in section 2). The metrics show 3 updated rows and 1 inserted row.
</div>

In [0]:
spark.sql(f"""
    MERGE INTO {target} t
    USING (SELECT customer_id, name, city, updated_at FROM changes) s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.table(target).orderBy("customer_id").show()
spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").select("operationMetrics").show(truncate=False)

## 2. Conditional clauses: deletes, and skipping unchanged rows

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Re-runs the merge with three refinements: delete rows flagged <code>D</code>, update only when something actually changed, and insert only non-delete rows.<br><br>
<b>Why it matters</b>
<ul><li>Unconditional <code>UPDATE SET *</code> rewrites rows that didn't change, which wastes work and, with change data feed, emits useless change events</li><li>CDC sources carry an operation flag, so <code>WHEN MATCHED AND s.op = 'D' THEN DELETE</code> applies deletes in the same transaction</li><li>Clauses are evaluated in order, so put the delete first</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C4 is deleted. C1 isn't updated because nothing changed. The metrics show 1 deleted row and 0 updated rows (C2 was already updated in section 1).
</div>

In [0]:
spark.sql(f"""
    MERGE INTO {target} t
    USING changes s
    ON t.customer_id = s.customer_id
    WHEN MATCHED AND s.op = 'D' THEN DELETE
    WHEN MATCHED AND (t.city <> s.city OR t.name <> s.name) THEN
      UPDATE SET t.name = s.name, t.city = s.city, t.updated_at = s.updated_at
    WHEN NOT MATCHED AND s.op <> 'D' THEN
      INSERT (customer_id, name, city, updated_at) VALUES (s.customer_id, s.name, s.city, s.updated_at)
""")
spark.table(target).orderBy("customer_id").show()
spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").select("operationMetrics").show(truncate=False)

## 3. The source must be unique per key

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Merges a source where C2 appears twice, sees the error, then deduplicates the source (keeping the latest row) and merges successfully.<br><br>
<b>Why it matters</b><br>If two source rows match the same target row, Delta can't know which update to apply, so the whole <code>MERGE</code> fails. CDC batches often contain several changes per key, so deduplicate with <code>row_number()</code> (lessons 01_10 and 01_19) first.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first merge fails with a <code>DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE</code> error. After deduplication, C2's latest city, Leeds, is applied.
</div>

In [0]:
from pyspark.sql import Window

dupes = spark.createDataFrame(
    [("C2", "John", "Liverpool", "2024-03-01"), ("C2", "John", "Leeds", "2024-03-05")],
    "customer_id STRING, name STRING, city STRING, updated_at STRING",
).withColumn("updated_at", F.to_date("updated_at"))
dupes.createOrReplaceTempView("dupes")

merge_sql = f"""
    MERGE INTO {target} t USING {{src}} s ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *
"""
try:
    spark.sql(merge_sql.format(src="dupes"))
except Exception as e:
    print("MERGE failed:", getattr(e, "getCondition", lambda: type(e).__name__)())

latest = dupes.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("updated_at")))) \
              .filter("rn = 1").drop("rn")
latest.createOrReplaceTempView("dupes_latest")
spark.sql(merge_sql.format(src="dupes_latest"))
spark.table(target).filter("customer_id = 'C2'").show()

## 4. Full sync with WHEN NOT MATCHED BY SOURCE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Treats a source as the complete list of active customers, and deletes target rows that are missing from it.<br><br>
<b>Why it matters</b><br>When a source sends a <b>full snapshot</b> rather than changes, <code>WHEN NOT MATCHED BY SOURCE</code> removes rows that no longer exist, so the target mirrors the source in one atomic operation. Be careful: run this only with a complete snapshot, or rows disappear by mistake.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After syncing with a snapshot containing only C1, C2 and C5, C3 is deleted. The metrics show <code>numTargetRowsNotMatchedBySourceDeleted = 1</code> (or a similar delete count).
</div>

In [0]:
snapshot = spark.table(target).filter("customer_id IN ('C1', 'C2', 'C5')")
snapshot.createOrReplaceTempView("snapshot")
spark.sql(f"""
    MERGE INTO {target} t USING snapshot s ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
spark.table(target).orderBy("customer_id").show()
spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").select("operationMetrics").show(truncate=False)

## 5. The Python API

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs an upsert with <code>DeltaTable.merge</code>.<br><br>
<b>Why it matters</b><br>The builder mirrors SQL clause by clause: <code>whenMatchedUpdate</code> / <code>whenMatchedUpdateAll</code>, <code>whenNotMatchedInsert</code> / <code>whenNotMatchedInsertAll</code>, <code>whenMatchedDelete</code>, and <code>whenNotMatchedBySourceDelete</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C6 inserted and C1's city updated to Bengaluru.
</div>

In [0]:
from delta.tables import DeltaTable

updates = spark.createDataFrame([("C1", "Amal", "Bengaluru", "2024-04-01"), ("C6", "Meena", "Kochi", "2024-04-01")],
                                "customer_id STRING, name STRING, city STRING, updated_at STRING").withColumn("updated_at", F.to_date("updated_at"))
(
    DeltaTable.forName(spark, target).alias("t")
    .merge(updates.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdateAll(condition="t.city <> s.city")
    .whenNotMatchedInsertAll()
    .execute()
)
spark.table(target).orderBy("customer_id").show()

## 6. SCD Type 2: keep the history

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a Type 2 customer dimension with <code>is_current</code>, <code>start_date</code> and <code>end_date</code>, then applies a change with the standard "staged updates" <code>MERGE</code> pattern.<br><br>
<b>Why it matters</b><br>For a changed customer, Type 2 must do <b>two</b> things to the same key: close the current row (update) <b>and</b> insert a new current row. A single <code>MERGE</code> can only do one thing per matched row, so the trick is to send the changed rows twice:
<ul><li>once with <code>merge_key = customer_id</code>, which matches and <b>closes</b> the current row</li><li>once with <code>merge_key = NULL</code>, which never matches, so it gets <b>inserted</b> as the new current row</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After the merge, C2 has two rows: London (closed, with <code>end_date = 2024-06-01</code>) and Glasgow (current). C7 is inserted as current. C1 is untouched.
</div>

In [0]:
dim = f"{DB}.dim_customer_scd2"
spark.sql(f"""
    CREATE OR REPLACE TABLE {dim} (
        customer_id STRING, city STRING, is_current BOOLEAN, start_date DATE, end_date DATE)
""")
spark.sql(f"""
    INSERT INTO {dim} VALUES
      ('C1', 'Chennai', true, DATE'2024-01-01', NULL),
      ('C2', 'London',  true, DATE'2024-01-01', NULL)
""")

spark.createDataFrame([("C1", "Chennai"), ("C2", "Glasgow"), ("C7", "Pune")], "customer_id STRING, city STRING") \
    .withColumn("effective_date", F.lit("2024-06-01").cast("date")).createOrReplaceTempView("scd_updates")

spark.sql(f"""
    MERGE INTO {dim} t
    USING (
        -- rows that will close the current version (only changed customers)
        SELECT u.customer_id AS merge_key, u.*
        FROM scd_updates u JOIN {dim} d
          ON u.customer_id = d.customer_id AND d.is_current AND u.city <> d.city
        UNION ALL
        -- rows that will be inserted as the new current version (changed + new customers)
        SELECT NULL AS merge_key, u.*
        FROM scd_updates u LEFT JOIN {dim} d
          ON u.customer_id = d.customer_id AND d.is_current
        WHERE d.customer_id IS NULL OR u.city <> d.city
    ) s
    ON t.customer_id = s.merge_key AND t.is_current
    WHEN MATCHED THEN
      UPDATE SET is_current = false, end_date = s.effective_date
    WHEN NOT MATCHED THEN
      INSERT (customer_id, city, is_current, start_date, end_date)
      VALUES (s.customer_id, s.city, true, s.effective_date, NULL)
""")
spark.table(dim).orderBy("customer_id", "start_date").show()

## 7. Querying a Type 2 dimension

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Gets the current city of each customer, and the city that was valid on a given date.<br><br>
<b>Why it matters</b><br>That's the payoff of Type 2: "as of now" and "as of then" from the same table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Current: C1 Chennai, C2 Glasgow, C7 Pune. As of 2024-03-15: C1 Chennai, C2 London (C7 didn't exist yet).
</div>

In [0]:
spark.sql(f"SELECT customer_id, city FROM {dim} WHERE is_current ORDER BY customer_id").show()
spark.sql(f"""
    SELECT customer_id, city FROM {dim}
    WHERE DATE'2024-03-15' >= start_date AND (end_date IS NULL OR DATE'2024-03-15' < end_date)
    ORDER BY customer_id
""").show()

## Under the hood

```
MERGE INTO target USING source ON cond
 1. join source with target (only files that may contain matching keys are read, using data skipping)
 2. classify each row: matched / not matched / not matched by source
 3. check: at most one source row per target row (else error)
 4. rewrite the touched target files (copy-on-write) or write deletion vectors + new rows
 5. commit: one version with remove/add actions and detailed metrics
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> `MERGE` joins source and target, so it shuffles unless the source is small enough to broadcast.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> one atomic commit.

**Performance tips:** keep the source small (only changes, deduplicated), add predicates that limit the target, such as `t.date >= '2024-06-01'`, so fewer files are read, cluster the target by the merge key, and use deletion vectors.

In [0]:
spark.sql(f"DESCRIBE HISTORY {target}").select("version", "operation", "operationMetrics.numTargetRowsUpdated", "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsDeleted").show()

In [0]:
for t in [target, dim]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE</code></b><br>
The source has duplicate keys. Deduplicate it first, keeping the latest row per key.<br><br>
<b>⛔ Problem: re-running the job duplicates rows</b><br>You used <code>append</code> instead of <code>MERGE</code>, or the <code>ON</code> condition doesn't match on the real business key.<br><br>
<b>⛔ Problem: everything is rewritten on every run</b><br>Unconditional <code>UPDATE SET *</code>. Add a "has anything changed" condition, and restrict the target with predicates.<br><br>
<b>⛔ Problem: rows deleted unexpectedly</b><br><code>WHEN NOT MATCHED BY SOURCE THEN DELETE</code> was used with a partial (incremental) source. Use it only with full snapshots.<br><br>
<b>⛔ Problem: SCD2 creates duplicates for unchanged rows</b><br>The staged source must include only <b>changed</b> or <b>new</b> keys. Filter <code>u.city &lt;&gt; d.city</code> (with null-safe comparisons in real code).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What does <code>MERGE INTO</code> do in Delta Lake?</b><br>
It matches source rows to target rows on a condition and, in a single atomic transaction, updates or deletes matched rows, inserts unmatched source rows, and optionally deletes target rows that have no source match. It's the standard way to do upserts and apply CDC.<br><br>

<b>🎤 2. How do you make an incremental load idempotent?</b><br>
Use <code>MERGE</code> on the business key instead of appending, with a deduplicated source. Running the same batch twice then updates the same rows to the same values instead of duplicating them.<br><br>

<b>🎤 3. What is the difference between SCD Type 1 and Type 2?</b><br>
Type 1 overwrites attributes with the latest values and keeps no history. Type 2 keeps every version as a separate row with validity dates and a current flag, so you can report on history.<br><br>

<b>🎤 4. How do you implement SCD Type 2 with <code>MERGE</code>?</b><br>
Stage the changed records twice: once with the real key, which matches the current row and closes it by setting <code>is_current</code> to false and the end date, and once with a null merge key, which never matches and is therefore inserted as the new current version. New keys only need the insert copy.<br><br>

<b>🎤 5. What happens if the source has two rows for the same key?</b><br>
The <code>MERGE</code> fails, because one target row can't be updated by multiple source rows. You deduplicate the source first, for example with <code>row_number()</code> ordered by timestamp.<br><br>

<b>🎤 6. How do you speed up a slow <code>MERGE</code>?</b><br>
Keep the source small and deduplicated, add target predicates such as date ranges so fewer files are scanned, cluster or Z-order the target by the merge key, broadcast a small source, avoid updating unchanged rows, and enable deletion vectors.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A daily batch of customer changes must update existing customers and insert new ones in a Delta table, without creating duplicates when the job is re-run. Which statement should be used?</b><br>
A. <code>INSERT INTO customers SELECT * FROM changes</code><br>
B. <code>MERGE INTO customers USING changes ON ... WHEN MATCHED THEN UPDATE ... WHEN NOT MATCHED THEN INSERT ...</code><br>
C. <code>INSERT OVERWRITE customers SELECT * FROM changes</code><br>
D. <code>COPY INTO customers FROM ...</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> C would delete every customer not present in the changes.</details><br><br>

<b>Q2. Which MERGE clause deletes target rows that do not exist in the source?</b><br>
A. <code>WHEN MATCHED THEN DELETE</code><br>
B. <code>WHEN NOT MATCHED THEN DELETE</code><br>
C. <code>WHEN NOT MATCHED BY SOURCE THEN DELETE</code><br>
D. <code>WHEN DELETED THEN REMOVE</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b></details><br><br>

<b>Q3. A <code>MERGE</code> fails because multiple source rows match the same target row. What is the correct fix?</b><br>
A. Add <code>ORDER BY</code> to the MERGE<br>
B. Deduplicate the source so each key appears once<br>
C. Run the MERGE twice<br>
D. Disable schema enforcement<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create <code>products (product_id, name, price, updated_at)</code> with 3 rows</li><li>Write a <code>MERGE</code> that applies a change set with an insert, a price change, a delete flag (<code>op = 'D'</code>), and an unchanged row, without updating the unchanged one</li><li>Make the change set contain two versions of one product, and deduplicate it before merging</li><li>Build an SCD Type 2 <code>dim_product_price</code> and apply two successive price changes for the same product. Show its full price history</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
products = f"{DB}.products_merge"

# 1. Target
spark.sql(f"CREATE OR REPLACE TABLE {products} (product_id INT, name STRING, price DOUBLE, updated_at DATE)")
spark.sql(f"INSERT INTO {products} VALUES (1, 'Laptop', 900.0, DATE'2024-01-01'), (2, 'Mouse', 20.0, DATE'2024-01-01'), (3, 'Desk', 250.0, DATE'2024-01-01')")

# 2-3. Change set with a duplicate key, deduplicated first
cs = spark.createDataFrame(
    [(1, "Laptop", 900.0, "2024-02-01", "U"), (2, "Mouse", 22.0, "2024-02-01", "U"), (2, "Mouse", 25.0, "2024-02-03", "U"),
     (3, "Desk", 250.0, "2024-02-01", "D"), (4, "Lamp", 35.0, "2024-02-01", "I")],
    "product_id INT, name STRING, price DOUBLE, updated_at STRING, op STRING").withColumn("updated_at", F.to_date("updated_at"))
cs = cs.withColumn("rn", F.row_number().over(Window.partitionBy("product_id").orderBy(F.desc("updated_at")))).filter("rn = 1").drop("rn")
cs.createOrReplaceTempView("product_changes")
spark.sql(f"""
    MERGE INTO {products} t USING product_changes s ON t.product_id = s.product_id
    WHEN MATCHED AND s.op = 'D' THEN DELETE
    WHEN MATCHED AND t.price <> s.price THEN UPDATE SET t.price = s.price, t.updated_at = s.updated_at
    WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT (product_id, name, price, updated_at) VALUES (s.product_id, s.name, s.price, s.updated_at)
""")
spark.table(products).orderBy("product_id").show()

# 4. SCD2 price history with two successive changes
dimp = f"{DB}.dim_product_price"
spark.sql(f"CREATE OR REPLACE TABLE {dimp} (product_id INT, price DOUBLE, is_current BOOLEAN, start_date DATE, end_date DATE)")
spark.sql(f"INSERT INTO {dimp} VALUES (2, 20.0, true, DATE'2024-01-01', NULL)")

def scd2_price(product_id, price, effective):
    spark.createDataFrame([(product_id, price)], "product_id INT, price DOUBLE") \
        .withColumn("effective_date", F.lit(effective).cast("date")).createOrReplaceTempView("price_updates")
    spark.sql(f"""
        MERGE INTO {dimp} t
        USING (
            SELECT u.product_id AS merge_key, u.* FROM price_updates u JOIN {dimp} d
              ON u.product_id = d.product_id AND d.is_current AND u.price <> d.price
            UNION ALL
            SELECT NULL AS merge_key, u.* FROM price_updates u LEFT JOIN {dimp} d
              ON u.product_id = d.product_id AND d.is_current
            WHERE d.product_id IS NULL OR u.price <> d.price
        ) s
        ON t.product_id = s.merge_key AND t.is_current
        WHEN MATCHED THEN UPDATE SET is_current = false, end_date = s.effective_date
        WHEN NOT MATCHED THEN INSERT (product_id, price, is_current, start_date, end_date)
                              VALUES (s.product_id, s.price, true, s.effective_date, NULL)
    """)

scd2_price(2, 22.0, "2024-02-01")
scd2_price(2, 25.0, "2024-03-01")
spark.table(dimp).orderBy("start_date").show()
for t in [products, dimp]:
    spark.sql(f"DROP TABLE {t}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>MERGE</code> = matched updates/deletes + unmatched inserts (+ not-matched-by-source deletes), in one atomic commit</li><li>The source must have at most one row per key. Deduplicate first</li><li>Add conditions to skip unchanged rows and to apply CDC deletes</li><li><code>WHEN NOT MATCHED BY SOURCE</code> only with complete snapshots</li><li>SCD1 = overwrite in place. SCD2 = close the current row + insert a new one (staged union with a null merge key)</li><li>Speed: small deduplicated source, target predicates, clustering by key, deletion vectors</li></ul>
</div>

| Clause | Use |
|---|---|
| `WHEN MATCHED [AND cond] THEN UPDATE SET * / SET a = s.a` | Update |
| `WHEN MATCHED [AND cond] THEN DELETE` | Delete matched |
| `WHEN NOT MATCHED [AND cond] THEN INSERT * / INSERT (...) VALUES (...)` | Insert new |
| `WHEN NOT MATCHED BY SOURCE [AND cond] THEN DELETE / UPDATE` | Handle rows missing from the source |
| Python | `DeltaTable.forName(...).alias("t").merge(src.alias("s"), cond).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()` |

## Git commit

```
git add 03_delta_lake/07_merge_upserts.ipynb
git commit -m "add 03_07 merge upserts notebook"
```